In [1]:
import torch
import torch.nn as nn
import cv2
import numpy as np
import math
from pathlib import Path
import matplotlib.pyplot as plt

In [2]:
InputDir = "TestDataset"
OutputDir = "TestDatasetRes"
ModelPath = "PrismModel.pth"
BucketsPath = "CoordBuckets.npy"
Temperature = 0.38

NumHeads = 8
HeadDims = 32
EmbedDim = 256

In [3]:
class Attention(nn.Module):
	def __init__(self):
		super().__init__()

		self.QueryMatrix = nn.Linear(EmbedDim,EmbedDim)
		self.KeyMatrix = nn.Linear(EmbedDim,EmbedDim)
		self.ValueMatrix = nn.Linear(EmbedDim,EmbedDim)
		self.OutputMatrix = nn.Linear(EmbedDim,EmbedDim)
		self.Softmax = nn.Softmax(dim=-1)

	def forward(self,x):
		# SeqLen = 256 currently
		BatchSize,SeqLen,_ = x.shape

		QueryOutput = self.QueryMatrix(x)
		KeyOutput = self.KeyMatrix(x)
		ValueOutput = self.ValueMatrix(x)

		# 8 Heads
		QueryOutput = QueryOutput.view(BatchSize,SeqLen,NumHeads,HeadDims).transpose(1,2) # B,NumHeads,256,HeadDims
		KeyOutput = KeyOutput.view(BatchSize,SeqLen,NumHeads,HeadDims).transpose(1,2) # B,NumHeads,256,HeadDims
		ValueOutput = ValueOutput.view(BatchSize,SeqLen,NumHeads,HeadDims).transpose(1,2) # B,NumHeads,256,HeadDims

		#Attention(Q, K, V) = (softmax((QK^T)/sqrt(dk)))*V, dk = HeadDims
		KeyOutput = KeyOutput.transpose(2,3) #B,NumHeads,HeadDims,SeqLen
		MatMulOutput = QueryOutput@KeyOutput #B,NumHeads,SeqLen,SeqLen
		MatMulOutput /= math.sqrt(HeadDims)
		MatMulOutput = self.Softmax(MatMulOutput)
		AttentionOutput = MatMulOutput@ValueOutput #B,NumHeads,SeqLen,HeadDims

		AttentionOutput = AttentionOutput.transpose(1,2).contiguous().view(BatchSize,SeqLen,EmbedDim)

		FinalOutput = self.OutputMatrix(AttentionOutput)

		return FinalOutput

In [4]:
class PrismBlock(nn.Module):
	def __init__(self):
		super().__init__()

		self.PseudoQuery = nn.Parameter(torch.randn(EmbedDim)*0.02)
		self.Attention = Attention()
		self.MultiLayerPerceptron = nn.Sequential(
			nn.Linear(in_features=EmbedDim,out_features=EmbedDim*4),
			nn.GELU(),
			nn.Linear(in_features=EmbedDim*4,out_features=EmbedDim)
		)
		self.LayerNorm1 = nn.LayerNorm(EmbedDim)
		self.LayerNorm2 = nn.LayerNorm(EmbedDim)
		self.LayerNorm3 = nn.LayerNorm(EmbedDim)
		self.Softmax = nn.Softmax(dim=0)

	def forward(self,x,HistoryList):
		HistoryList = torch.stack(HistoryList,dim=0) # D,B,SeqLen,EmbedDim
		HistoryListNorm = self.LayerNorm1(HistoryList)

		Scores = (HistoryListNorm*self.PseudoQuery).sum(dim=-1)/math.sqrt(EmbedDim) # D,B,SeqLen
		Weights = self.Softmax(Scores).unsqueeze(-1)

		XRouted = (HistoryList*Weights).sum(dim=0)

		XRoutedNorm = self.LayerNorm2(XRouted)
		x = x + self.Attention(XRoutedNorm)

		xNorm = self.LayerNorm3(x)
		x = x + self.MultiLayerPerceptron(xNorm)

		return x

In [5]:
class PrismModel(nn.Module):
	def __init__(self):
		super().__init__()
		self.HighResConv = nn.Sequential(
			nn.Conv2d(in_channels=1,out_channels=64,kernel_size=3,stride=1,padding=1),
			nn.GELU()
		)
		self.InputLayer = nn.Conv2d(in_channels=64,out_channels=256,kernel_size=16,stride=16) #B,256,16,16
		self.Flatten = nn.Flatten(start_dim=2) #B,256,256
		#Transpose to B,256,256
		self.LearnParam = nn.Parameter(torch.randn((1,256,256)) * 0.02)
		self.ModuleList = nn.ModuleList(
			[PrismBlock() for _ in range(8)]
		)
		self.FinalNorm = nn.LayerNorm(EmbedDim)
		self.Decoder1 = nn.Sequential(
			# 16x16 -> 32x32
			nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
			nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1),
			nn.GroupNorm(32,256),
			nn.GELU(),
			# 32x32 -> 64x64
			nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
			nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1),
			nn.GroupNorm(32,256),
			nn.GELU(),
			# 64x64 -> 128x128
			nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
			nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1),
			nn.GroupNorm(32,256),
			nn.GELU()
		)
		self.Decoder2 = nn.Sequential(
			# 128x128 -> 256x256
			nn.Conv2d(in_channels=320, out_channels=128, kernel_size=3, stride=1, padding=1),
			nn.GroupNorm(16,128),
			nn.GELU(),
			# Final map
			nn.Conv2d(in_channels=128, out_channels=313, kernel_size=3, stride=1, padding=1)
		)

	def forward(self,x):
		SkipFeatures = self.HighResConv(x) # B,64,256,256
		x = self.InputLayer(SkipFeatures) # B,256,16,16
		x = self.Flatten(x) # B,256,256
		x = x.transpose(1,2)
		x = x + self.LearnParam # B,256,256

		HistoryList = [x]
		for module in self.ModuleList:
			x = module(x,HistoryList)
			HistoryList.append(x)

		x = self.FinalNorm(x)

		x = x.transpose(1,2).reshape(-1,256,16,16)

		x = self.Decoder1(x)

		x = nn.functional.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)

		x = torch.cat([x,SkipFeatures], dim=1) # B,320,256,256

		x = self.Decoder2(x)

		return x

In [6]:
def CleanCheckpointStateDict(RawStateDict):
	CleanStateDict = {}
	for key, value in RawStateDict.items():
		CleanKey = key.replace('_orig_mod.', '').replace('module.', '')
		CleanStateDict[CleanKey] = value
	return CleanStateDict

def ColorizeSingleImage(ImagePath, model, BucketsTensor, device):
	ImgBGR = cv2.imread(str(ImagePath))
	if ImgBGR is None:
		print(f"Skipping {ImagePath}, OpenCV could not read it.")
		return None, None
	
	OrigH, OrigW = ImgBGR.shape[:2]
	ImgLAB = cv2.cvtColor(ImgBGR, cv2.COLOR_BGR2LAB)
	LOrig, _, _ = cv2.split(ImgLAB)

	LResized = cv2.resize(LOrig, (256, 256), interpolation=cv2.INTER_CUBIC)
	InputTensor = (torch.from_numpy(LResized).float() / 255.0).unsqueeze(0).unsqueeze(0).to(device)

	with torch.no_grad():
		with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu', dtype=torch.float16):
			Logits = model(InputTensor)
		
		Logits = Logits.float()
		
		Probs = torch.nn.functional.softmax(Logits / Temperature, dim=1)
		PredAB = torch.einsum('b c h w, c d -> b d h w', Probs, BucketsTensor)

	PredABNumpy = PredAB.squeeze(0).cpu().numpy().transpose(1, 2, 0)
	PredABResized = cv2.resize(PredABNumpy, (OrigW, OrigH), interpolation=cv2.INTER_CUBIC)

	PredA = np.clip(PredABResized[..., 0] + 128.0, 0, 255).astype(np.uint8)
	PredB = np.clip(PredABResized[..., 1] + 128.0, 0, 255).astype(np.uint8)

	MergedLAB = cv2.merge([LOrig, PredA, PredB])
	OutputBGR = cv2.cvtColor(MergedLAB, cv2.COLOR_LAB2BGR)

	return ImgBGR, OutputBGR

def ProcessDirectory():
	device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
	print(f"Initializing inference on {device}")

	model = PrismModel().to(device)
	Checkpoint = torch.load(ModelPath, map_location=device, weights_only=False)
	StateDict = Checkpoint.get('ModelState', Checkpoint)
	model.load_state_dict(CleanCheckpointStateDict(StateDict))
	model.eval()

	Buckets = np.load(BucketsPath)
	BucketsTensor = torch.from_numpy(Buckets).float().to(device)

	InPath = Path(InputDir)
	OutPath = Path(OutputDir)
	OutPath.mkdir(parents=True, exist_ok=True)
	
	InputPaths = [p for p in InPath.glob('*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
	if not InputPaths:
		print(f"No valid images found in {InputDir}")
		return

	print(f"Found {len(InputPaths)} images.")

	for idx, ImgPath in enumerate(InputPaths, 1):
		print(f"[{idx}/{len(InputPaths)}] Colorizing {ImgPath.name}")
		OrigBGR, OutputBGR = ColorizeSingleImage(ImgPath, model, BucketsTensor, device)
		
		if OrigBGR is None:
			continue
		
		ComparisonImage = cv2.hconcat([OrigBGR, OutputBGR])
		
		cv2.putText(ComparisonImage, "Original", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
		cv2.putText(ComparisonImage, "Prism", (OrigBGR.shape[1] + 10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
		
		PlotPath = OutPath / f"Comparison{ImgPath.name}"
		cv2.imwrite(str(PlotPath), ComparisonImage)
		
		del OrigBGR, OutputBGR, ComparisonImage

In [7]:
if __name__ == '__main__':
	ProcessDirectory()

Initializing inference on cuda
Found 6489 images.
[1/6489] Colorizing flowers_019922.jpg
[2/6489] Colorizing animals_009967.jpg
[3/6489] Colorizing indoor_047819.jpg
[4/6489] Colorizing humans_042585.jpg
[5/6489] Colorizing food_027560.jpg
[6/6489] Colorizing humans_041315.jpg
[7/6489] Colorizing animals_009250.jpg
[8/6489] Colorizing humans_041244.jpg
[9/6489] Colorizing ocean_053598.jpg
[10/6489] Colorizing night_049128.jpg
[11/6489] Colorizing buildings_013864.jpg
[12/6489] Colorizing food_027311.jpg
[13/6489] Colorizing vehicles_064378.jpg
[14/6489] Colorizing humans_041462.jpg
[15/6489] Colorizing humans_041501.jpg
[16/6489] Colorizing scenery_057735.jpg
[17/6489] Colorizing indoor_047474.jpg
[18/6489] Colorizing indoor_047576.jpg
[19/6489] Colorizing vehicles_064720.jpg
[20/6489] Colorizing humans_042275.jpg
[21/6489] Colorizing flowers_019995.jpg
[22/6489] Colorizing vehicles_064744.jpg
[23/6489] Colorizing flowers_020515.jpg
[24/6489] Colorizing scenery_057678.jpg
[25/6489] Col